In [44]:
from transformers import AutoTokenizer
import tiktoken
import os
import pandas as pd

In [45]:
TESTS = {
    "english" :    "I am looking into the tokenized inputs from different models.",
    "telugu":      "సర్వర్ సాధారణంగా పనిచేస్తోంది.",
    "code":        "def calculate_score(x): return x ** 2 + 1",
    "long_number": "The invoice total is 8394572103.",
    "email":       "durgasrinivasdonkina@gmail.com",
    "url":         "https://docs.vllm.ai/en/latest/configuration/engine_args.html",
    "math":        "Solve for x: 3x² - 7x + 2 = 0",   # your actual domain
}

HF = {
    "Qwen3.5": "Qwen/Qwen3.5-2B",    
    "Llama-3":  "NousResearch/Meta-Llama-3-8B",   
    "bert" : "google-bert/bert-base-uncased"
}

In [46]:
rows = []
tokenizers = {a : AutoTokenizer.from_pretrained(b) for a,b in HF.items()}
gpt4o = tiktoken.encoding_for_model("gpt-4o")

In [47]:
for label, text in TESTS.items():
    row = {"text": label, "chars": len(text), "words": len(text.split())}
    for name, tok in tokenizers.items():
        row[name] = len(tok.encode(text, add_special_tokens=False))
        row[f"{name}_roundtrip_ok"]  = tok.decode(tok.encode(text, add_special_tokens=False)) == text
    row["GPT-4o"] = len(gpt4o.encode(text))
    row[f"gpt4o_roundtrip_ok"]  = gpt4o.decode(gpt4o.encode(text)) == text
    rows.append(row)

print(pd.DataFrame(rows).to_string(index=False))

       text  chars  words  Qwen3.5  Qwen3.5_roundtrip_ok  Llama-3  Llama-3_roundtrip_ok  bert  bert_roundtrip_ok  GPT-4o  gpt4o_roundtrip_ok
    english     61     10       12                  True       12                  True    12              False      12                True
     telugu     30      3       18                  True       55                  True     4              False      10                True
       code     41      8       13                  True       13                  True    15              False      13                True
long_number     32      5       16                  True       10                  True    14              False      10                True
      email     30      1        9                  True       10                  True    12              False       9                True
        url     61      1       13                  True       13                  True    25              False      15                True
       math  

In [48]:
qwen = tokenizers["Qwen3.5"]
for label in ["long_number", "math", "telugu"]:
    ids = tok.encode(TESTS[label], add_special_tokens=False)
    print(f"\n {label} : {tok.convert_ids_to_tokens(ids)}")


 long_number : ['the', 'in', '##vo', '##ice', 'total', 'is', '83', '##9', '##45', '##7', '##21', '##0', '##3', '.']

 math : ['solve', 'for', 'x', ':', '3', '##x', '##²', '-', '7', '##x', '+', '2', '=', '0']

 telugu : ['[UNK]', '[UNK]', '[UNK]', '.']


In [49]:
Llama = tokenizers["Llama-3"]
for label in ["long_number", "math", "telugu"]:
    ids = Llama.encode(TESTS[label], add_special_tokens=False)
    print(f"\n {label} : {Llama.convert_ids_to_tokens(ids)}")


 long_number : ['The', 'Ġinvoice', 'Ġtotal', 'Ġis', 'Ġ', '839', '457', '210', '3', '.']

 math : ['S', 'olve', 'Ġfor', 'Ġx', ':', 'Ġ', '3', 'x', 'Â²', 'Ġ-', 'Ġ', '7', 'x', 'Ġ+', 'Ġ', '2', 'Ġ=', 'Ġ', '0']

 telugu : ['à°', '¸', 'à°', '°', 'à±', 'į', 'à°', 'µ', 'à°', '°', 'à±', 'į', 'Ġà°', '¸', 'à°', '¾', 'à°', '§', 'à°', '¾', 'à°', '°', 'à°', '£', 'à°', 'Ĥ', 'à°', 'Ĺ', 'à°', '¾', 'Ġà°', 'ª', 'à°', '¨', 'à°', '¿', 'à°', 'ļ', 'à±', 'ĩ', 'à°', '¸', 'à±', 'į', 'à°', '¤', 'à±', 'ĭ', 'à°', 'Ĥ', 'à°', '¦', 'à°', '¿', '.']


In [50]:
bert = tokenizers["bert"]
print(bert.tokenize(TESTS["url"]))
print(bert.tokenize(TESTS["english"]))

['https', ':', '/', '/', 'doc', '##s', '.', 'v', '##ll', '##m', '.', 'ai', '/', 'en', '/', 'latest', '/', 'configuration', '/', 'engine', '_', 'ar', '##gs', '.', 'html']
['i', 'am', 'looking', 'into', 'the', 'token', '##ized', 'inputs', 'from', 'different', 'models', '.']


In [51]:
text = TESTS["english"]
print(bert.tokenize(text))
print(bert.decode(bert.encode(text, add_special_tokens=False)))


['i', 'am', 'looking', 'into', 'the', 'token', '##ized', 'inputs', 'from', 'different', 'models', '.']
i am looking into the tokenized inputs from different models.


In [52]:
text = TESTS["telugu"]
print(qwen.tokenize(text))
print(qwen.decode(qwen.encode(text, add_special_tokens=False)))
print(qwen.encode(text, add_special_tokens=False))

['à°¸', 'à°°à±į', 'à°µ', 'à°°à±į', 'Ġà°¸', 'à°¾', 'à°§', 'à°¾à°°', 'à°£', 'à°Ĥà°Ĺà°¾', 'Ġà°ª', 'à°¨à°¿', 'à°ļ', 'à±ĩ', 'à°¸à±įà°¤', 'à±ĭ', 'à°Ĥà°¦à°¿', '.']
సర్వర్ సాధారణంగా పనిచేస్తోంది.
[151327, 173254, 152051, 173254, 154040, 148846, 169236, 152561, 169349, 197343, 155525, 155551, 150885, 151337, 202461, 150515, 178183, 13]


In [53]:
text = TESTS['math']
print(bert.tokenize(text))
print(qwen.tokenize(text))


['solve', 'for', 'x', ':', '3', '##x', '##²', '-', '7', '##x', '+', '2', '=', '0']
['S', 'olve', 'Ġfor', 'Ġx', ':', 'Ġ', '3', 'x', 'Â²', 'Ġ-', 'Ġ', '7', 'x', 'Ġ+', 'Ġ', '2', 'Ġ=', 'Ġ', '0']


In [54]:
for label, text in TESTS.items():
    print(f"QWEN : {label} : {qwen.tokenize(text)} : { len(qwen.tokenize(text))}")
    

QWEN : english : ['I', 'Ġam', 'Ġlooking', 'Ġinto', 'Ġthe', 'Ġtoken', 'ized', 'Ġinputs', 'Ġfrom', 'Ġdifferent', 'Ġmodels', '.'] : 12
QWEN : telugu : ['à°¸', 'à°°à±į', 'à°µ', 'à°°à±į', 'Ġà°¸', 'à°¾', 'à°§', 'à°¾à°°', 'à°£', 'à°Ĥà°Ĺà°¾', 'Ġà°ª', 'à°¨à°¿', 'à°ļ', 'à±ĩ', 'à°¸à±įà°¤', 'à±ĭ', 'à°Ĥà°¦à°¿', '.'] : 18
QWEN : code : ['def', 'Ġcalculate', '_score', '(x', '):', 'Ġreturn', 'Ġx', 'Ġ**', 'Ġ', '2', 'Ġ+', 'Ġ', '1'] : 13
QWEN : long_number : ['The', 'Ġinvoice', 'Ġtotal', 'Ġis', 'Ġ', '8', '3', '9', '4', '5', '7', '2', '1', '0', '3', '.'] : 16
QWEN : email : ['d', 'urg', 'as', 'rin', 'ivas', 'don', 'kina', '@gmail', '.com'] : 9
QWEN : url : ['https', '://', 'docs', '.v', 'll', 'm', '.ai', '/en', '/latest', '/configuration', '/engine', '_args', '.html'] : 13
QWEN : math : ['S', 'olve', 'Ġfor', 'Ġx', ':', 'Ġ', '3', 'x', 'Â²', 'Ġ-', 'Ġ', '7', 'x', 'Ġ+', 'Ġ', '2', 'Ġ=', 'Ġ', '0'] : 19


In [55]:
for label, text in TESTS.items():
    print(f"Llama : {label} : {Llama.tokenize(text)} : { len( Llama.tokenize(text))}")

Llama : english : ['I', 'Ġam', 'Ġlooking', 'Ġinto', 'Ġthe', 'Ġtoken', 'ized', 'Ġinputs', 'Ġfrom', 'Ġdifferent', 'Ġmodels', '.'] : 12
Llama : telugu : ['à°', '¸', 'à°', '°', 'à±', 'į', 'à°', 'µ', 'à°', '°', 'à±', 'į', 'Ġà°', '¸', 'à°', '¾', 'à°', '§', 'à°', '¾', 'à°', '°', 'à°', '£', 'à°', 'Ĥ', 'à°', 'Ĺ', 'à°', '¾', 'Ġà°', 'ª', 'à°', '¨', 'à°', '¿', 'à°', 'ļ', 'à±', 'ĩ', 'à°', '¸', 'à±', 'į', 'à°', '¤', 'à±', 'ĭ', 'à°', 'Ĥ', 'à°', '¦', 'à°', '¿', '.'] : 55
Llama : code : ['def', 'Ġcalculate', '_score', '(x', '):', 'Ġreturn', 'Ġx', 'Ġ**', 'Ġ', '2', 'Ġ+', 'Ġ', '1'] : 13
Llama : long_number : ['The', 'Ġinvoice', 'Ġtotal', 'Ġis', 'Ġ', '839', '457', '210', '3', '.'] : 10
Llama : email : ['d', 'urg', 'as', 'rin', 'ivas', 'don', 'k', 'ina', '@gmail', '.com'] : 10
Llama : url : ['https', '://', 'docs', '.v', 'll', 'm', '.ai', '/en', '/latest', '/configuration', '/engine', '_args', '.html'] : 13
Llama : math : ['S', 'olve', 'Ġfor', 'Ġx', ':', 'Ġ', '3', 'x', 'Â²', 'Ġ-', 'Ġ', '7', 'x', 'Ġ+', 'Ġ', 

In [56]:
for label, text in TESTS.items():
    print(f" BERT : {label} : {bert.tokenize(text)} : {len(bert.tokenize(text))}")

 BERT : english : ['i', 'am', 'looking', 'into', 'the', 'token', '##ized', 'inputs', 'from', 'different', 'models', '.'] : 12
 BERT : telugu : ['[UNK]', '[UNK]', '[UNK]', '.'] : 4
 BERT : code : ['def', 'calculate', '_', 'score', '(', 'x', ')', ':', 'return', 'x', '*', '*', '2', '+', '1'] : 15
 BERT : long_number : ['the', 'in', '##vo', '##ice', 'total', 'is', '83', '##9', '##45', '##7', '##21', '##0', '##3', '.'] : 14
 BERT : email : ['durga', '##sr', '##ini', '##vas', '##don', '##kin', '##a', '@', 'gma', '##il', '.', 'com'] : 12
 BERT : url : ['https', ':', '/', '/', 'doc', '##s', '.', 'v', '##ll', '##m', '.', 'ai', '/', 'en', '/', 'latest', '/', 'configuration', '/', 'engine', '_', 'ar', '##gs', '.', 'html'] : 25
 BERT : math : ['solve', 'for', 'x', ':', '3', '##x', '##²', '-', '7', '##x', '+', '2', '=', '0'] : 14


In [57]:
# Observations : 

# 1. why does token count differ from word count ? 

# Ans : Because of BPE ( Byte-Pair Encoding ), Tokenizers does not tokenize words based on meaning but rather based on character. BPE repeatedly merges the most frequent adjancent pair in the corpus . Basically each tokenizer or model has a fixed vocab, the number of tokens per sentence is negatively correlated to that of vocab size. If that word is not present in the vocab, it is basically divided into sub word tokenizing , thus causing more token count than the atcual word count.

# 2. Why is you multilingual bot more expensive?

# Ans: Because if the tokenziers vocab is trained on english heavy corpus, non english words are hard to merge and make tokens resulting in each character as token most of time, thus causing more token count over less words in a sentence. More tokens means more spending as billing is per token.

# 3. why do models fumble long numbers?

# Ans : Because some tokenizers divide the numbers into fixed numbered groups, converting the long number into a fixed number groups , while regrouping effectively loosing the place value like if groups are inter arranged. this issue is effectively reduced by tokenizing each number individually, to preserve the value of number.



In [58]:
# 1. Why does token count differ from word count?

# Tokenizers split text using BPE (Byte-Pair Encoding), which is statistical, not semantic — it doesn't split on word meaning. BPE starts from bytes/characters and repeatedly merges the most frequent adjacent pair in the training corpus until it reaches a fixed vocabulary size. Common words end up as a single token (often with the leading space bundled in, so " the" is one token), while rare or unseen words are broken into multiple subword tokens. Token count is negatively correlated with vocab size — a larger vocab means more words fit as single tokens, so fewer splits — but with diminishing returns, not a strict inverse. Net effect: frequent words ≈ 1 token, rare words > 1 token, so total token count usually exceeds word count (English averages ~1.3 tokens/word).

# 2. Why is your multilingual bot more expensive?

# Because the tokenizer's vocabulary is trained on an English-heavy corpus. English text got the most frequent merges, so it tokenizes efficiently (~1.3 tokens/word). Non-English scripts (Telugu, Hindi) appear far less in training, so few merges were learned for them — they fragment down to near-character or near-byte level, producing many tokens for a single word. This is compounded by UTF-8 encoding: Latin characters are 1 byte, but Telugu/Devanagari characters are 3 bytes each, giving BPE more pieces to work with in the first place. The result is the same sentence costing 2–4× more tokens in these languages. Since billing, latency, and context-window usage are all measured per token (not per byte), more tokens directly means higher cost.

# 3. Why do models fumble long numbers?

# Primarily because of how tokenizers chunk digits. Some tokenizers (e.g. older GPT-4/cl100k) group digits into up-to-3-digit tokens with inconsistent boundaries, so 1234 might split as 123|4 but 12345 as 123|45. This misaligns place value — the model can't reliably tell which token holds the hundreds vs. thousands place — which corrupts arithmetic. Tokenizing each digit individually (as Llama does) fixes the alignment, since every digit is a clean, consistent token. So tokenization determines how badly a given model struggles. However, it isn't the whole story: even with clean digit tokenization, transformers still make arithmetic errors because operations like carrying and long multiplication are hard for the architecture itself. Tokenization is the largest single factor, not the only one.